# 美债账户监控测试 notebook

本 notebook 分两部分：

1. 默认离线测试：使用 mock IB 对象验证美债过滤、持仓明细、Greeks、图表和 what-if 容量计算。
2. 可选在线测试：手动设置 `RUN_LIVE_IB = True` 后连接 IB Gateway/TWS 拉取真实账户快照。

默认不会连接 IB，也不会发送任何订单。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import math
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "target_treasury_account_monitor").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from target_treasury_account_monitor.config import DEFAULT_CLIENT_ID, DEFAULT_HOST, DEFAULT_MARKET_DATA_LABEL, DEFAULT_PORT, DEFAULT_REFRESH_SECONDS, MARKET_DATA_TYPES, MonitorSettings
from target_treasury_account_monitor.contracts import is_treasury_contract
from target_treasury_account_monitor.frames import excluded_positions_frame, positions_to_frame
from target_treasury_account_monitor.greeks import greek_totals
from target_treasury_account_monitor.margin import estimate_contract_capacity, order_state_to_margin_row

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. 构造离线 mock 数据

这里模拟一个账户里同时有 ZF 期货、ZF 期货期权和一条非美债股票持仓。

In [ ]:
ACCOUNT = "MOCK"

future_contract = SimpleNamespace(
    conId=1,
    secType="FUT",
    symbol="ZF",
    localSymbol="ZFM6",
    tradingClass="ZF",
    exchange="CBOT",
    currency="USD",
    lastTradeDateOrContractMonth="202606",
    strike=0,
    right="",
    multiplier="1000",
)
option_contract = SimpleNamespace(
    conId=2,
    secType="FOP",
    symbol="ZF",
    localSymbol="OZF JUN 106.75P",
    tradingClass="ZF",
    exchange="CBOT",
    currency="USD",
    lastTradeDateOrContractMonth="20260615",
    strike=106.75,
    right="P",
    multiplier="1000",
)
stock_contract = SimpleNamespace(
    conId=3,
    secType="STK",
    symbol="AAPL",
    localSymbol="AAPL",
    tradingClass="AAPL",
    exchange="SMART",
    currency="USD",
    lastTradeDateOrContractMonth="",
    strike=0,
    right="",
    multiplier="1",
)

positions = [
    SimpleNamespace(account=ACCOUNT, contract=future_contract, position=2, avgCost=106.20),
    SimpleNamespace(account=ACCOUNT, contract=option_contract, position=-5, avgCost=0.09),
    SimpleNamespace(account=ACCOUNT, contract=stock_contract, position=100, avgCost=180.0),
]

future_ticker = SimpleNamespace(
    contract=future_contract,
    bid=106.50,
    ask=106.55,
    last=106.52,
    markPrice=math.nan,
    close=106.40,
    delayedBid=math.nan,
    delayedAsk=math.nan,
    delayedLast=math.nan,
    delayedClose=math.nan,
    modelGreeks=None,
    lastGreeks=None,
    askGreeks=None,
    bidGreeks=None,
    impliedVolatility=math.nan,
    marketPrice=lambda: 106.52,
)
option_ticker = SimpleNamespace(
    contract=option_contract,
    bid=0.10,
    ask=0.12,
    last=0.11,
    markPrice=math.nan,
    close=0.09,
    delayedBid=math.nan,
    delayedAsk=math.nan,
    delayedLast=math.nan,
    delayedClose=math.nan,
    impliedVolatility=0.19,
    modelGreeks=SimpleNamespace(impliedVol=0.18, delta=-0.35, gamma=0.04, theta=-0.012, vega=0.22, optPrice=0.11, undPrice=106.90),
    lastGreeks=None,
    askGreeks=None,
    bidGreeks=None,
    marketPrice=lambda: math.nan,
)

tickers = {1: future_ticker, 2: option_ticker}
portfolio_map = {
    1: SimpleNamespace(account=ACCOUNT, contract=future_contract, marketPrice=106.52, marketValue=213040.0, unrealizedPNL=640.0, realizedPNL=0.0),
    2: SimpleNamespace(account=ACCOUNT, contract=option_contract, marketPrice=0.11, marketValue=-550.0, unrealizedPNL=-100.0, realizedPNL=0.0),
}
summary = pd.DataFrame([
    {"account": ACCOUNT, "tag": "NetLiquidation", "value": 100000.0, "currency": "USD"},
    {"account": ACCOUNT, "tag": "ExcessLiquidity", "value": 80000.0, "currency": "USD"},
    {"account": ACCOUNT, "tag": "AvailableFunds", "value": 76000.0, "currency": "USD"},
    {"account": ACCOUNT, "tag": "InitMarginReq", "value": 12000.0, "currency": "USD"},
    {"account": ACCOUNT, "tag": "MaintMarginReq", "value": 10000.0, "currency": "USD"},
])

## 2. 验证美债过滤、明细表和 Greeks

In [ ]:
treasury_positions = [pos for pos in positions if is_treasury_contract(pos.contract)]
assert len(treasury_positions) == 2
assert not is_treasury_contract(stock_contract)

frame = positions_to_frame(treasury_positions, tickers, portfolio_map)
display(frame[["optionName", "secType", "position", "price", "marketValue", "iv", "delta", "gamma", "theta", "vega", "missingData"]])

future_delta = frame.loc[frame["secType"] == "FUT", "delta"].iloc[0]
option_delta = frame.loc[frame["secType"] == "FOP", "delta"].iloc[0]
assert future_delta == 1.0
assert round(option_delta, 2) == -0.35

totals = greek_totals(frame)
display(totals)
assert not totals.empty
assert "deltaMultiplier" in totals.columns

excluded = excluded_positions_frame(positions)
display(excluded)
assert excluded.iloc[0]["symbol"] == "AAPL"

## 3. 验证可视化图表对象

这里只检查图表能正常生成；在 notebook 中显示第一张 Greeks 图。

In [ ]:
try:
    from target_treasury_account_monitor.visualization import chart_greek_exposure, chart_liquidity, chart_position_market_value, chart_unrealized_pnl
except ModuleNotFoundError as exc:
    print(f"缺少可视化依赖，已跳过图表测试：{exc}")
else:
    charts = [
        chart_greek_exposure(frame),
        chart_liquidity(summary),
        chart_position_market_value(frame),
        chart_unrealized_pnl(frame),
    ]
    assert all(chart is not None for chart in charts)
    display(charts[0])

## 4. 验证 what-if 保证金容量计算

这里不连接 IB，只模拟 IB what-if 返回的 OrderState。真实试算请用 Streamlit 页面或 `test_margin_whatif.py`。

In [ ]:
mock_order_state = SimpleNamespace(
    initMarginBefore="12,000",
    initMarginChange="1,200",
    initMarginAfter="13,200",
    maintMarginBefore="10,000",
    maintMarginChange="1,000",
    maintMarginAfter="11,000",
    equityWithLoanBefore="100,000",
    equityWithLoanChange="0",
    equityWithLoanAfter="100,000",
    commission="2.50",
    minCommission="1.00",
    maxCommission="5.00",
    warningText="",
)
margin_row = order_state_to_margin_row(mock_order_state)
capacity_row = estimate_contract_capacity(summary, margin_row, safety_buffer=5000)
result = pd.DataFrame([{**margin_row, **capacity_row}])
display(result[["initMarginChange", "maintMarginChange", "bindingMarginChange", "usableLiquidity", "estimatedMaxContracts"]])
assert capacity_row["bindingMarginChange"] == 1200.0
assert capacity_row["estimatedMaxContracts"] == 62

## 5. 可选：连接 IB 拉取真实快照

确认 IB Gateway/TWS 已经启动并开启 API 后，把下面的 `RUN_LIVE_IB` 改成 `True`，再填写真实账户。

In [ ]:
RUN_LIVE_IB = False
TARGET_ACCOUNT = ""
HOST = DEFAULT_HOST
PORT = DEFAULT_PORT
CLIENT_ID = DEFAULT_CLIENT_ID + 20
MARKET_DATA = DEFAULT_MARKET_DATA_LABEL  # 没有实时 CBOT 行情权限时，默认用 Delayed 避免 354 刷屏
QUOTE_WAIT_SECONDS = 8.0

In [ ]:
if RUN_LIVE_IB:
    from ib_async import IB, util
    from ib_async.ib import StartupFetch
    from target_treasury_account_monitor.ib_client import subscribe_quotes_for_positions
    from target_treasury_account_monitor.snapshot import build_snapshot

    assert TARGET_ACCOUNT, "请先填写 TARGET_ACCOUNT"
    settings = MonitorSettings(
        host=HOST,
        port=PORT,
        client_id=CLIENT_ID,
        account=TARGET_ACCOUNT,
        market_data_type=MARKET_DATA_TYPES[MARKET_DATA],
        quote_wait_seconds=QUOTE_WAIT_SECONDS,
        refresh_seconds=DEFAULT_REFRESH_SECONDS,
        auto_refresh=False,
        auto_reconnect=False,
        reconnect_backoff_seconds=10,
        wechat_webhook_url="",
        wechat_push_enabled=False,
        wechat_min_interval_seconds=300,
        order_preview_enabled=False,
        readonly=True,
    )

    util.startLoop()
    ib = IB()
    ib.connect(settings.host, settings.port, clientId=settings.client_id, readonly=settings.readonly, timeout=10, fetchFields=StartupFetch.POSITIONS | StartupFetch.ACCOUNT_UPDATES | StartupFetch.SUB_ACCOUNT_UPDATES)
    try:
        snapshot = build_snapshot(ib, settings, lambda positions: subscribe_quotes_for_positions(ib, positions, settings))
        print("updated_at:", snapshot.updated_at)
        print("treasury positions:", len(snapshot.positions), "/ all positions:", len(snapshot.all_positions))
        display(snapshot.frame.head(50))
        display(greek_totals(snapshot.frame))
    finally:
        ib.disconnect()
else:
    print("RUN_LIVE_IB=False，已跳过在线 IB 测试。")

## 6. 可选：一分钟动态刷新演示

真实监控建议直接使用 Streamlit 页面。下面的单元只用于 notebook 中手动抽查刷新逻辑。

In [ ]:
RUN_DYNAMIC_DEMO = False
DYNAMIC_ROUNDS = 3

if RUN_DYNAMIC_DEMO:
    import time
    from ib_async import IB, util
    from ib_async.ib import StartupFetch
    from target_treasury_account_monitor.ib_client import subscribe_quotes_for_positions
    from target_treasury_account_monitor.snapshot import build_snapshot

    assert TARGET_ACCOUNT, "请先填写 TARGET_ACCOUNT"
    settings = MonitorSettings(
        host=HOST,
        port=PORT,
        client_id=CLIENT_ID + 1,
        account=TARGET_ACCOUNT,
        market_data_type=MARKET_DATA_TYPES[MARKET_DATA],
        quote_wait_seconds=QUOTE_WAIT_SECONDS,
        refresh_seconds=DEFAULT_REFRESH_SECONDS,
        auto_refresh=True,
        auto_reconnect=False,
        reconnect_backoff_seconds=10,
        wechat_webhook_url="",
        wechat_push_enabled=False,
        wechat_min_interval_seconds=300,
        order_preview_enabled=False,
        readonly=True,
    )

    util.startLoop()
    ib = IB()
    ib.connect(settings.host, settings.port, clientId=settings.client_id, readonly=settings.readonly, timeout=10, fetchFields=StartupFetch.POSITIONS | StartupFetch.ACCOUNT_UPDATES | StartupFetch.SUB_ACCOUNT_UPDATES)
    try:
        ticker_cache = {"tickers": None}

        def load_quotes(positions):
            ticker_cache["tickers"] = subscribe_quotes_for_positions(
                ib,
                positions,
                settings,
                previous_tickers=ticker_cache["tickers"],
            )
            return ticker_cache["tickers"]

        for i in range(DYNAMIC_ROUNDS):
            snapshot = build_snapshot(ib, settings, load_quotes)
            print(f"round={i + 1}, updated_at={snapshot.updated_at}, rows={len(snapshot.frame)}")
            display(snapshot.frame[["optionName", "position", "price", "delta", "gamma", "theta", "vega", "missingData"]].head(20))
            if i + 1 < DYNAMIC_ROUNDS:
                time.sleep(DEFAULT_REFRESH_SECONDS)
    finally:
        ib.disconnect()
else:
    print("RUN_DYNAMIC_DEMO=False，已跳过一分钟动态刷新演示。")